In [57]:
#import the library pulp as p
import pulp as p
from pulp import *
import pandas as pd
import numpy as np
import math
from scipy.stats import norm
import pickle

In [58]:
with open('values_Nicole_exp.pkl','rb') as file: #loading the values from the pickle file
    values1=pickle.load(file)
print(values1)

{'Strike range': [[1, 3], [0, 4]], 'No. of discretization points': 400, 'Upper bound': 12.543787812520153, 'Epsilon': np.float64(0.015037593984962405)}


In [59]:
N = 8 #values1['No. of discretization points']
# K1_inf = values1['Strike range']

#strike_bounds = [ val for tup in values1['Strike range'] for val in tup]
strike_ranges = [[90,125],[90,125]] #values1['Strike range']
print(f"Strike ranges:{strike_ranges}")

# print(strike_bounds)
strike_bound1 = [90,125]#strike_bounds[:2]
strike_bound2 = [90,125]#strike_bounds[2:]

print(f"1st truncation range:{strike_bound1}")
print(f"2nd truncation range:{strike_bound2}")

Strike ranges:[[90, 125], [90, 125]]
1st truncation range:[90, 125]
2nd truncation range:[90, 125]


In [60]:
# #inputing the parameter values

# #spacing between strike points/discretization points is 1/n
# N = 100


stock = 110
K = 120
r = 0
sigma = 1
T = 1
mu=0
t = 0
timepoints = 2 #number of timsteps
# #t1 = np.linspace(0,T,num=(timepoints+1))
t1 = [0,0.5,T]
# #number of short maturity options
no_of_options = 2
print("timesteps:", t1)
Epsilon = 0

timesteps: [0, 0.5, 1]


In [61]:
#first choice of strike prices
#K1 = np.linspace(strike_bound1[0],strike_bound1[-1],N)
K1 = np.linspace(strike_bound1[0],strike_bound1[-1],N)
print(f"The first set of strike prices:{K1}")
#the spacing between each consecutive element in K1 is (K1[N-1]-K1[0])/(N-1)
print(np.diff(K1)[0])
print(f"spacing:{(K1[-1]-K1[0])/(N-1)}")
#second choice of strike prices
K2 = np.linspace(strike_bound2[0],strike_bound2[-1],N)
print(f"The second set of strike prices:{K2}")
#getting the spacing between each consecutive element in K1
print(np.diff(K2)[0])
print(f"spacing:{(K2[-1]-K2[0])/(N-1)}")


The first set of strike prices:[ 90.  95. 100. 105. 110. 115. 120. 125.]
5.0
spacing:5.0
The second set of strike prices:[ 90.  95. 100. 105. 110. 115. 120. 125.]
5.0
spacing:5.0


In [62]:
# # #function to get European call price
# def BS_call(a,b,strike):
#     if a <= strike <=b:
#         #print(f"strike {strike} in range {[a,b]}")
#         call = np.divide(1,b-a) * ( 0.5 * (b**2) - b * strike + 0.5 * (strike**2))
#     elif 0 <= strike < a :
#         #print(f"strike {strike} less than lower bound {1}")      
#         call = np.divide(1,b-a) * ( 0.5 * (b**2 - a **2) - (b -a) * strike )
#     else:
#         call = 0
#     return call
# print(BS_call(1,3,0))

In [63]:


call_prices_0=[2.2825,1.78,1.265,0.78,0.345,0.06,0.025,0.01]
call_prices_1=[2.405,1.907,1.414,0.976,0.613,0.365,0.2005,0.11]
call_prices = np.vstack((call_prices_0,call_prices_1))
print(f"The call prices are:{call_prices}")

The call prices are:[[2.2825 1.78   1.265  0.78   0.345  0.06   0.025  0.01  ]
 [2.405  1.907  1.414  0.976  0.613  0.365  0.2005 0.11  ]]


In [64]:
disc_points = np.zeros((timepoints,N))
disc_points[0] = K1
disc_points[1] = K2
#print(f"Discretization points : {disc_points}")

In [65]:


#generate the new marginals
#function to get the marginal distributions
#timepoints denote the number of timepoints at which the marginals are getting calculated

def marginal_new(call_prices_array_new,disc_points_new,timepoints,t1):
    
    marg = np.zeros_like(disc_points_new)
    
    for k in range(1,timepoints+1):
        print("timepoint:",t1[k])
        sum1 = 0
        N = len(disc_points_new[k-1])
        print(N)
        for i in range(N):
            if i == 0:
                #\mu(k_j) = (C(k_{j+1})-C(k_{j}))/(k_{j+1}-k_{j})+1
                marg[k-1,i] = np.divide(call_prices_array_new[k-1,i+1]-call_prices_array_new[k-1,i],\
                                          disc_points_new[k-1,i+1]-  disc_points_new[k-1,i])+1
            elif i == N-1:
                #mu_{k_{n})=-(C(k_{n})-C(k_{n-1}))/(k_{n}-k_{n-1})
                marg[k-1,i] = -np.divide(call_prices_array_new[k-1,i]-call_prices_array_new[k-1,i-1],\
                                       disc_points_new[k-1,i]-disc_points_new[k-1,i-1])
            
            else:
                #\mu(k_j) = (C(k_{j+1})-C(k_{j}))/(k_{j+1}-k_{j})-(C(k_{j})-C(k_{j-1}))/(k_{j}-k_{j-1})
                marg[k-1,i] = np.divide(call_prices_array_new[k-1,i+1]-call_prices_array_new[k-1,i],\
                                          disc_points_new[k-1,i+1]-  disc_points_new[k-1,i])\
                            - np.divide(call_prices_array_new[k-1,i]-call_prices_array_new[k-1,i-1],\
                                       disc_points_new[k-1,i]-disc_points_new[k-1,i-1])
               
            
            #marg[k-1,i] = np.round(marg[k-1,i],8)
            #marg[k-1,i] = max(marg[k-1,i],0)
            sum1 += marg[k-1,i]    
        print("Sum of marginals at",str(t1[k]),"is:",sum(marg[k-1,:]))
    return marg

marg = marginal_new(call_prices,disc_points,timepoints,t1)
print(marg)

timepoint: 0.5
8
Sum of marginals at 0.5 is: 1.0
timepoint: 1
8
Sum of marginals at 1 is: 1.0
[[ 0.8995 -0.0025  0.006   0.01    0.03    0.05    0.004   0.003 ]
 [ 0.9004  0.001   0.011   0.015   0.023   0.0167  0.0148  0.0181]]


In [66]:
marg[0][0],marg[0][1] = 0.895,0.002
print(f"The updated marginals are:{marg}")

The updated marginals are:[[0.895  0.002  0.006  0.01   0.03   0.05   0.004  0.003 ]
 [0.9004 0.001  0.011  0.015  0.023  0.0167 0.0148 0.0181]]


In [67]:
#strike points, which also act as stock prices at intermediate time points
K1 = disc_points[0]
K2 = disc_points[1]
# print("Strike points at time",str(t1[1]),"are:", K1)
# print("Strike points at time",str(t1[2]),"are:",K2)

In [68]:
#model initialisation
model = LpProblem("Upper-Bound-Problem", LpMaximize)
# Create an LP Maximization problem
Lp_prob1 = p.LpProblem('Problem', p.LpMaximize)


# Create problem Variables

n_timepoint_1 = n_timepoint_2 = N

#j for second time point and i for first time point
variable_names = [(str(i),str(j))  for i in range(n_timepoint_1) for j in range(n_timepoint_2)]
#pd.DataFrame(variable_names).replace(',', '').astype(int)
#variable_names.sort(key=lambda x: x)
#print("Variable Indices:", variable_names)

DV_variables = LpVariable.matrix("X", (range(n_timepoint_1),range(n_timepoint_2)), lowBound= 0, upBound=1  )
#DV_variables.sort(key=lambda X:(X.name))

allocation = np.array(DV_variables).reshape(N,N)
#print("Decision Variable/Allocation Matrix: ")
#print(allocation)



# slack_names = [(int(i)) for i in range(n_timepoint_1)]
# print("Slack Indices:", slack_names)
# Slack_variables = LpVariable.matrix("Delta", range(n_timepoint_1), lowBound= 0)
# #Slack_variables.sort(key=lambda Delta:Delta.name)
# delta = np.array(Slack_variables).reshape(N,1)
# print("Slack variables:")
# print(delta)

In [69]:
#payoff for Asian option
def payoff_asian(X,Y):
    #F = max(0.5*(X+Y)-K,0)
    F =max(0.5*(X+Y)-K,0)
    return F

In [70]:
#formulating the objective function

#cost function
#cost = [multiperiod_payoff(K1[i],K2[j],no_of_options,K) for i in range(0,N) for j in range(0,N)]
cost = [payoff_asian(K1[i],K2[j]) for i in range(0,N) for j in range(0,N)]
#objective function
cost_matrix = np.array(cost).reshape(N,N)
print("cost:", cost_matrix)

for i in range(0,N):
    for j in range(0,N):
        if payoff_asian(K1[i],K2[j]) != cost_matrix[i,j]:
            print(f"{i},{j} violated")
    
obj_func = lpSum(allocation*cost_matrix)
#print(obj_func)
model +=  obj_func
#print(model)

cost: [[0.  0.  0.  0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.  0.  0.  0.  2.5]
 [0.  0.  0.  0.  0.  0.  2.5 5. ]]


In [71]:
#time point 1 Constraints
for i in range(n_timepoint_1):
    #print(lpSum(allocation[i][j] for j in range(n_timepoint_2)) == marg[0][i])
    model += lpSum(allocation[i][j] for j in range(n_timepoint_2)) == marg[0][i] , "time 1 Constraint " + str(i)
    

In [72]:
#time point 2 Constraints
for j in range(n_timepoint_2):
    #print(lpSum(allocation[i][j] for i in range(n_timepoint_1)) == marg[1][j])
    model += lpSum(allocation[i][j] for i in range(n_timepoint_1)) == marg[1][j] , "time 2 Constraint " + str(j)

In [73]:
#print("cost:", cost_matrix)

In [74]:
#epsilon-martingale constraints
#time point 1 Constraints with delta

for i in range(n_timepoint_1):
    #\sum_{j}p(i,j)*(y_j -x_i) <= delta_i
    
   # print(lpSum(allocation[i][j]*(K2[j] - K1[i]) for j in range(n_timepoint_2))\
    #    == 0 )
    model += lpSum(allocation[i][j]*(K2[j] - K1[i]) for j in range(n_timepoint_2))\
         == 0 ,\
    " Time 1 Martingale Constraint " + str(i+1)

In [75]:
# #epsilon-martingale constraints
# #time point 2 Constraints

# for i in range(n_timepoint_1):
#     #sum_{j} p(i,j)(y_j -x_i) +delta_i >=0
    
#     print(0<=\
#           lpSum(allocation[i][j]*(K2[j] - K1[i]) for j in range(n_timepoint_2))\
#           +delta[i])
#     model += 0<=\
#           lpSum(allocation[i][j]*(K2[j] - K1[i]) for j in range(n_timepoint_2))\
#           +delta[i],\
#     "Time 2 Epsilon-Martingale Constraint " + str(i+1)
    

In [76]:
# #time 1 delta constraint

# #sum_{i} delta_i <= epsilon_n

# print(lpSum(delta[i] for i in range(n_timepoint_1)) -Epsilon <=0)

# model += lpSum(delta[i] for i in range(n_timepoint_1)) -Epsilon <=0,\
#     "Delta Constraint " 

In [77]:
#print(model)

In [78]:
#model.solve()
model.solve(PULP_CBC_CMD())
status =  LpStatus[model.status]


print(status)

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /home/jupyter-purba/.local/lib/python3.9/site-packages/pulp/solverdir/cbc/linux/64/cbc /tmp/98ff1b09eb974fffaa03d7c488af8ca3-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /tmp/98ff1b09eb974fffaa03d7c488af8ca3-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 29 COLUMNS
At line 217 RHS
At line 242 BOUNDS
At line 307 ENDATA
Problem MODEL has 24 rows, 64 columns and 184 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 18 (-6) rows, 46 (-18) columns and 142 (-42) elements
Perturbing problem by 0.001% of 2.5 - largest nonzero change 0.00021216789 ( 0.0084867154%) - largest zero change 0.00016301659
0  Obj 0.05275 Primal inf 0.19540267 (14)
16  Obj 0.023564725
Optimal - objective value 0.023571429
After Postsolve, objective 0.023571429, infeasibilities - dual 0 (0), primal 0 (0)
Optimal objectiv

In [79]:
print(f"status: {model.status}, {LpStatus[model.status]}")

status: 1, Optimal


In [80]:
#model.solve(PULP_CBC_CMD(msg=1))

In [81]:
#output the optimal values of the objective function and the decision variables
print("Upper bound:", model.objective.value())
call_value = model.objective.value()
# Decision Variables


for v in model.variables():
    try:
        print(v.name,"=", v.value())
        
    except:
        print("error couldnt find value")

Upper bound: 0.023571428499999998
X_0_0 = 0.895
X_0_1 = 0.0
X_0_2 = 0.0
X_0_3 = 0.0
X_0_4 = 0.0
X_0_5 = 0.0
X_0_6 = 0.0
X_0_7 = 0.0
X_1_0 = 0.0017142857
X_1_1 = 0.0
X_1_2 = 0.0
X_1_3 = 0.0
X_1_4 = 0.0
X_1_5 = 0.0
X_1_6 = 0.0
X_1_7 = 0.00028571429
X_2_0 = 0.0
X_2_1 = 0.0
X_2_2 = 0.006
X_2_3 = 0.0
X_2_4 = 0.0
X_2_5 = 0.0
X_2_6 = 0.0
X_2_7 = 0.0
X_3_0 = 0.0
X_3_1 = 0.001
X_3_2 = 0.0
X_3_3 = 0.0083333333
X_3_4 = 0.0
X_3_5 = 0.0
X_3_6 = 0.00066666667
X_3_7 = 0.0
X_4_0 = 0.0023333333
X_4_1 = 0.0
X_4_2 = 0.0
X_4_3 = 0.0
X_4_4 = 0.023
X_4_5 = 0.0
X_4_6 = 0.0046666667
X_4_7 = 0.0
X_5_0 = 0.00078095238
X_5_1 = 0.0
X_5_2 = 0.005
X_5_3 = 0.0066666667
X_5_4 = 0.0
X_5_5 = 0.0167
X_5_6 = 0.0094666667
X_5_7 = 0.011385714
X_6_0 = 0.00057142857
X_6_1 = 0.0
X_6_2 = 0.0
X_6_3 = 0.0
X_6_4 = 0.0
X_6_5 = 0.0
X_6_6 = 0.0
X_6_7 = 0.0034285714
X_7_0 = 0.0
X_7_1 = 0.0
X_7_2 = 0.0
X_7_3 = 0.0
X_7_4 = 0.0
X_7_5 = 0.0
X_7_6 = 0.0
X_7_7 = 0.003


In [82]:
for name, constraint in model.constraints.items():
    print(f"{name}: {constraint.value()}") 

time_1_Constraint_0: 0.0
time_1_Constraint_1: -9.999999960041972e-12
time_1_Constraint_2: -4.683753385137379e-17
time_1_Constraint_3: -2.999996572775748e-11
time_1_Constraint_4: -1.3010426069826053e-17
time_1_Constraint_5: -2.200000043250938e-10
time_1_Constraint_6: -2.999999901276418e-11
time_1_Constraint_7: 0.0
time_2_Constraint_0: -5.000007124913303e-11
time_2_Constraint_1: 6.852157730108388e-17
time_2_Constraint_2: -3.729655473350135e-17
time_2_Constraint_3: 1.474514954580286e-17
time_2_Constraint_4: 0.0
time_2_Constraint_5: 0.0
time_2_Constraint_6: 7.000000058765554e-11
time_2_Constraint_7: -3.100000022307481e-10
_Time_1_Martingale_Constraint_1: 0.0
_Time_1_Martingale_Constraint_2: 1.9999999920083944e-10
_Time_1_Martingale_Constraint_3: 0.0
_Time_1_Martingale_Constraint_4: 5.00000006675716e-11
_Time_1_Martingale_Constraint_5: 9.999999925347502e-10
_Time_1_Martingale_Constraint_6: -3.000000026176508e-09
_Time_1_Martingale_Constraint_7: -9.999999786569624e-11
_Time_1_Martingale_Cons

In [83]:
#calculating Asian call value for discrete arithmetic average using Monte Carlo simulations
import random
def Asian_call_MC(stock, K, t1, t2):
    tau = t1 
    tau2 = t2 - t1
    M = 1000
    S1 = np.zeros((M))
    S2 = np.zeros((M))
    random.seed(10)
    sum1 = 0
    for i in range(M):
        #simulating stock paths at time u
        S1[i] = stock * np.exp((r - 0.5 * sigma**2) * tau + sigma * np.sqrt(tau) * random.gauss(0,1))
        
        #simulating stock paths at time v
        S2[i] = S1[i] * np.exp((r - 0.5 * sigma**2) * tau2 + sigma * np.sqrt(tau2) * random.gauss(0,1))
            
        #payoff for the hedge portfolio with short maturity options 
        #sum1 += multiperiod_payoff(S1[i],S2[i],Strike_1,Strike_2,weight_1,weight_2,K)
        sum1 += payoff_asian( S1[i], S2[i])
        #sum1 += max(0.5*(S1[i]+S2[i])-K,0)
    call = np.exp(-r * t2) * (sum1/M)
    return call

print("Value of the Asian call using MC is:", Asian_call_MC(stock,K,t1[1],t1[2]))
print(t1[1],t1[2]) 
      
        
        

Value of the Asian call using MC is: 29.97416185193128
0.5 1


In [84]:
#perc_error = np.divide(Asian_call_MC(stock,K,t1[1],t1[2])- call_value, Asian_call_MC(stock,K,t1[1]\
                # ,t1[2]))*100
perc_error = np.divide(12.5- call_value,12.5)*100  # np.divide(12.5- call_value, 12.5)*100                                                                                   
print("Upper bound:", call_value) 

print("percentage error:",perc_error,"%")

# D1 = np.arange(0,R_m +1/n,1/n)

# print("Number of discretization points:", len(D))

# #we need one extra strike point to calculate the marginals
# print("Number of call prices required:", len(D1))

Upper bound: 0.023571428499999998
percentage error: 99.811428572 %


In [85]:
print(f"status: {model.status}, {LpStatus[model.status]}")

status: 1, Optimal


In [86]:
# #extracting the indices and variable values
# import re
# index_1=[]
# index_2=[]

# Pr=np.zeros((N,N))

# for i in range(N*N):
#     v = model.variables()[i]
#     try:
#         print(v.name,"=", v.value())
#         v1 = v.name.split("_",2)
#         # The string
#         text = v.name
#         val = v.value()
        
#         # Regular expression pattern to match numbers within single quotes
#         pattern = r"'(\d+)'\s*,_\s*'(\d+)'"

#         # Find all matches
#         matches = re.search(pattern, text)
#         if matches:
#                # Extract the numbers
#             number1 = matches.group(1)
#             number2 = matches.group(2)
            
#             # Print the extracted numbers
#             print(number1)  
#             print(number2)  
            
#             index_1.append(number1)
#             index_2.append(number2)
            
#             Pr[int(number1)][int(number2)] = val
            
            

#     except:
#         print("error couldnt find value")



In [87]:
# Print the variable names and their values
for v in DV_variables:
    for var in v:  # v is a list of variables for each row
        print(var.name, "=", value(var))

X_0_0 = 0.895
X_0_1 = 0.0
X_0_2 = 0.0
X_0_3 = 0.0
X_0_4 = 0.0
X_0_5 = 0.0
X_0_6 = 0.0
X_0_7 = 0.0
X_1_0 = 0.0017142857
X_1_1 = 0.0
X_1_2 = 0.0
X_1_3 = 0.0
X_1_4 = 0.0
X_1_5 = 0.0
X_1_6 = 0.0
X_1_7 = 0.00028571429
X_2_0 = 0.0
X_2_1 = 0.0
X_2_2 = 0.006
X_2_3 = 0.0
X_2_4 = 0.0
X_2_5 = 0.0
X_2_6 = 0.0
X_2_7 = 0.0
X_3_0 = 0.0
X_3_1 = 0.001
X_3_2 = 0.0
X_3_3 = 0.0083333333
X_3_4 = 0.0
X_3_5 = 0.0
X_3_6 = 0.00066666667
X_3_7 = 0.0
X_4_0 = 0.0023333333
X_4_1 = 0.0
X_4_2 = 0.0
X_4_3 = 0.0
X_4_4 = 0.023
X_4_5 = 0.0
X_4_6 = 0.0046666667
X_4_7 = 0.0
X_5_0 = 0.00078095238
X_5_1 = 0.0
X_5_2 = 0.005
X_5_3 = 0.0066666667
X_5_4 = 0.0
X_5_5 = 0.0167
X_5_6 = 0.0094666667
X_5_7 = 0.011385714
X_6_0 = 0.00057142857
X_6_1 = 0.0
X_6_2 = 0.0
X_6_3 = 0.0
X_6_4 = 0.0
X_6_5 = 0.0
X_6_6 = 0.0
X_6_7 = 0.0034285714
X_7_0 = 0.0
X_7_1 = 0.0
X_7_2 = 0.0
X_7_3 = 0.0
X_7_4 = 0.0
X_7_5 = 0.0
X_7_6 = 0.0
X_7_7 = 0.003


In [88]:
#joint probability matrix
Pr=np.zeros((N,N))
for i in range(n_timepoint_1):
        for j in range(n_timepoint_2):
            Pr[i][j] = DV_variables[i][j].value()
print(Pr)
print(Pr[2][1])

[[8.9500000e-01 0.0000000e+00 0.0000000e+00 0.0000000e+00 0.0000000e+00
  0.0000000e+00 0.0000000e+00 0.0000000e+00]
 [1.7142857e-03 0.0000000e+00 0.0000000e+00 0.0000000e+00 0.0000000e+00
  0.0000000e+00 0.0000000e+00 2.8571429e-04]
 [0.0000000e+00 0.0000000e+00 6.0000000e-03 0.0000000e+00 0.0000000e+00
  0.0000000e+00 0.0000000e+00 0.0000000e+00]
 [0.0000000e+00 1.0000000e-03 0.0000000e+00 8.3333333e-03 0.0000000e+00
  0.0000000e+00 6.6666667e-04 0.0000000e+00]
 [2.3333333e-03 0.0000000e+00 0.0000000e+00 0.0000000e+00 2.3000000e-02
  0.0000000e+00 4.6666667e-03 0.0000000e+00]
 [7.8095238e-04 0.0000000e+00 5.0000000e-03 6.6666667e-03 0.0000000e+00
  1.6700000e-02 9.4666667e-03 1.1385714e-02]
 [5.7142857e-04 0.0000000e+00 0.0000000e+00 0.0000000e+00 0.0000000e+00
  0.0000000e+00 0.0000000e+00 3.4285714e-03]
 [0.0000000e+00 0.0000000e+00 0.0000000e+00 0.0000000e+00 0.0000000e+00
  0.0000000e+00 0.0000000e+00 3.0000000e-03]]
0.0


In [89]:
#
#print(Pr)
print(Pr.shape)
#print(len(index_1))

(8, 8)


In [90]:
MM = []
for i in range(N):
    diff = 0
    for j in range(N):
        diff += Pr[i,j]*(K2[j]-K1[i]) 
    #print("Difference between epsilon and martingale constraint", i,"is:",Epsilon-diff)
    MM.append(abs(diff))
print(max(MM))
    

3.000000026176508e-09


In [91]:
print(f"status: {model.status}, {LpStatus[model.status]}")
#print("Strike range:", [0,R_m])
print("No. of discretization points:", N)
print("Upper bound:", call_value) 
print("Epsilon:", Epsilon)
# print("Epsilon using max of martingale constraints:",N*round(max(MM),4))
# print(round(max(MM),4))
print("percentage error:",perc_error,"%")

status: 1, Optimal
No. of discretization points: 8
Upper bound: 0.023571428499999998
Epsilon: 0
percentage error: 99.811428572 %


In [92]:
# import pickle
# values = {"Strike range":[K1[0],K1[-1]],"spacing, n":n,"Sigma":sigma,"No. of discretization points":N,"Upper bound":call_value,\
#           "max(MM)":max(MM),"Epsilon":N*round(max(MM),4)}

# with open('values_UG.pkl', 'wb') as file: #writing as a pickle file
#     pickle.dump(values, file)

In [93]:
# with open('values_UG.pkl','rb') as file: #loading the values from the pickle file
#     values1=pickle.load(file)
# print(values1)

In [96]:
print(sum(marg[1][i]*max(disc_points[1][i]-K,0) for i in range(len(disc_points[1]))))
print(sum(Pr[i,j]*K2[j] for i in range(N)for j in range(N)))


0.09050000000000001
92.29499996514997
